## Downloading the Data

In [1]:
!kaggle competitions download -c nlp-getting-started


  0%|          | 0.00/593k [00:00<?, ?B/s]
100%|██████████| 593k/593k [00:00<00:00, 3.59MB/s]
100%|██████████| 593k/593k [00:00<00:00, 3.57MB/s]


## Extracting the Data

In [2]:
import zipfile
import os

os.mkdir('data')

In [3]:
zipref = zipfile.ZipFile('nlp-getting-started.zip', 'r')

In [4]:
zipref.extractall('data')

In [5]:
zipref.close()

## Loading the Data

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [7]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [8]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Null Values

In [9]:
train.isna().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [10]:
test.isna().sum()

id             0
keyword       26
location    1105
text           0
dtype: int64

In [11]:
train["keyword"].nunique()

221

In [12]:
train["location"].nunique()

3341

In [15]:
keywords = train["keyword"].unique()
keywords

array([nan, 'ablaze', 'accident', 'aftershock', 'airplane%20accident',
       'ambulance', 'annihilated', 'annihilation', 'apocalypse',
       'armageddon', 'army', 'arson', 'arsonist', 'attack', 'attacked',
       'avalanche', 'battle', 'bioterror', 'bioterrorism', 'blaze',
       'blazing', 'bleeding', 'blew%20up', 'blight', 'blizzard', 'blood',
       'bloody', 'blown%20up', 'body%20bag', 'body%20bagging',
       'body%20bags', 'bomb', 'bombed', 'bombing', 'bridge%20collapse',
       'buildings%20burning', 'buildings%20on%20fire', 'burned',
       'burning', 'burning%20buildings', 'bush%20fires', 'casualties',
       'casualty', 'catastrophe', 'catastrophic', 'chemical%20emergency',
       'cliff%20fall', 'collapse', 'collapsed', 'collide', 'collided',
       'collision', 'crash', 'crashed', 'crush', 'crushed', 'curfew',
       'cyclone', 'damage', 'danger', 'dead', 'death', 'deaths', 'debris',
       'deluge', 'deluged', 'demolish', 'demolished', 'demolition',
       'derail', 'der

In [18]:
locations = train["location"].unique()
locations[:20]

array([nan, 'Birmingham', 'Est. September 2012 - Bristol', 'AFRICA',
       'Philadelphia, PA', 'London, UK', 'Pretoria', 'World Wide!!',
       'Paranaque City', 'Live On Webcam', 'milky way',
       'GREENSBORO,NORTH CAROLINA', 'England.',
       'Sheffield Township, Ohio', 'India', 'Barbados', 'Anaheim',
       'Abuja', 'USA', 'South Africa'], dtype=object)

In [14]:
train['target'].value_counts()

0    4342
1    3271
Name: target, dtype: int64

In [35]:
for i in range(10):
    print(train['text'][i])

Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Forest fire near La Ronge Sask. Canada
All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected
13,000 people receive #wildfires evacuation orders in California 
Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school 
#RockyFire Update => California Hwy. 20 closed in both directions due to Lake County fire - #CAfire #wildfires
#flood #disaster Heavy rain causes flash flooding of streets in Manitou, Colorado Springs areas
I'm on top of the hill and I can see a fire in the woods...
There's an emergency evacuation happening now in the building across the street
I'm afraid that the tornado is coming to our area...


In [30]:
labelzero = train[train['target'] == 0]
labelzero.head()

,id,keyword,location,text,target
15,23,NaN,NaN,What's up man?,0
16,24,NaN,NaN,I love fruits,0
17,25,NaN,NaN,Summer is lovely,0
18,26,NaN,NaN,My car is so fast,0
19,28,NaN,NaN,What a goooooooaaaaaal!!!!!!,0


In [34]:
for i in range(15, 25):
    print(labelzero['text'][i])

What's up man?
I love fruits
Summer is lovely
My car is so fast
What a goooooooaaaaaal!!!!!!
this is ridiculous....
London is cool ;)
Love skiing
What a wonderful day!
LOOOOOOL


### Cleaning the Data

In [60]:
from nltk.corpus import stopwords
def clean(text):
    text = text.lower()

    chars = ["!", "#", ",", ".", ":", ";", "?", "-", "@", "~", ")", "(", "=", ">", "<", "/"]
    for char in chars:
        if char in text:
            text = text.replace(char, "")
    
    stop = stopwords.words('english')
    text = [word for word in text.split() if word not in stop]
    return " ".join(text)

In [61]:
text = """#RockyFire Update => California Hwy. 20 closed in both directions due to Lake County fire - #CAfire #wildfires"""
clean(text=text)

'rockyfire update california hwy 20 closed directions due lake county fire cafire wildfires'

In [63]:
train['text'] = train['text'].apply(clean)
test['text'] = test['text'].apply(clean)

In [65]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=1000)
cv.fit(train['text'])
cvr = cv.transform(train['text'])
cvr.shape

(7613, 1000)

In [ ]:
X = cvr.toarray()
y = train['target']

In [70]:
cv = CountVectorizer(max_features=1000)
cv.fit(test['text'])
cvr = cv.transform(test['text'])
cvr.shape
X_test = cvr.toarray()

In [71]:
X_test.shape

(3263, 1000)

In [72]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [73]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

In [74]:
lr.score(X_train, y_train)

0.8509031198686371

In [75]:
lr.score(X_val, y_val)

0.7912015758371634

In [97]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, max_depth = 20)
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=20, n_estimators=200)

In [98]:
rf.score(X_train, y_train)

0.7394088669950739

In [99]:
rf.score(X_val, y_val)

0.7124097176625082

In [104]:
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators=200)
xgb.fit(X_train, y_train)

C:\Users\harik\anaconda3\envs\data_sciences\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[14:17:31] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, enable_categorical=False,
              gamma=0, gpu_id=-1, importance_type=None,
              interaction_constraints='', learning_rate=0.300000012,
              max_delta_step=0, max_depth=6, min_child_weight=1, missing=nan,
              monotone_constraints='()', n_estimators=200, n_jobs=8,
              num_parallel_tree=1, predictor='auto', random_state=0,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, subsample=1,
              tree_method='exact', validate_parameters=1, verbosity=None)

In [105]:
rf.score(X_val, y_val)

0.7124097176625082

In [106]:
rf.score(X_train, y_train)

0.7394088669950739

In [107]:
X_train.shape

(6090, 1000)

In [119]:
import tensorflow.keras.layers as layers
import tensorflow as tf
model = tf.keras.models.Sequential([
    # layers.Dense(64, activation='relu'),
    layers.Dense(128, activation='relu', input_shape=(1000,)),
    layers.Dropout(0.2),
    layers.BatchNormalization(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),

    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.BatchNormalization(),

    # layers.Dense(1024, activation='relu'),
    # layers.Dropout(0.5),
    # layers.BatchNormalization(),

    layers.Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential_5"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_33 (Dense)             (None, 128)               128128    
_________________________________________________________________
dropout_19 (Dropout)         (None, 128)               0         
_________________________________________________________________
batch_normalization_7 (Batch (None, 128)               512       
_________________________________________________________________
dense_34 (Dense)             (None, 256)               33024     
_________________________________________________________________
dropout_20 (Dropout)         (None, 256)               0         
_________________________________________________________________
batch_normalization_8 (Batch (None, 256)               1024      
_________________________________________________________________
dense_35 (Dense)             (None, 512)              

In [120]:
model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

Epoch 1/10
191/191 [==============================] - 5s 14ms/step - loss: 0.7100 - accuracy: 0.6425 - val_loss: 0.6649 - val_accuracy: 0.5982
Epoch 2/10
191/191 [==============================] - 2s 10ms/step - loss: 0.4992 - accuracy: 0.7745 - val_loss: 0.5131 - val_accuracy: 0.7498
Epoch 3/10
191/191 [==============================] - 2s 13ms/step - loss: 0.4090 - accuracy: 0.8296 - val_loss: 0.5162 - val_accuracy: 0.7656
Epoch 4/10
191/191 [==============================] - 2s 12ms/step - loss: 0.3589 - accuracy: 0.8452 - val_loss: 0.5218 - val_accuracy: 0.7827
Epoch 5/10
191/191 [==============================] - 2s 12ms/step - loss: 0.3106 - accuracy: 0.8726 - val_loss: 0.5474 - val_accuracy: 0.7794
Epoch 6/10
191/191 [==============================] - 2s 12ms/step - loss: 0.2675 - accuracy: 0.8944 - val_loss: 0.5948 - val_accuracy: 0.7728
Epoch 7/10
191/191 [==============================] - 3s 13ms/step - loss: 0.2414 - accuracy: 0.9046 - val_loss: 0.6706 - val_accuracy: 0.7656

In [128]:
lr.score(X_val, y_val)

0.7912015758371634

In [129]:
preds = lr.predict(X_test)

In [130]:
preds = (preds>0.5).astype(int)
preds

array([1, 1, 1, ..., 0, 0, 0])

In [131]:
sample = pd.read_csv('data/sample_submission.csv')
sample['target'] = preds
sample.to_csv('submission.csv', index=False)

In [132]:
sample['target'].value_counts()

0    2058
1    1205
Name: target, dtype: int64

In [133]:
!kaggle competitions submit -c nlp-getting-started -f submission.csv -m "Baseline"

Successfully submitted to Natural Language Processing with Disaster Tweets



  0%|          | 0.00/25.4k [00:00<?, ?B/s]
 31%|███▏      | 8.00k/25.4k [00:00<00:00, 65.0kB/s]
100%|██████████| 25.4k/25.4k [00:07<00:00, 3.71kB/s]
